In [1]:
# ======================================================================
# DUBLIN LOCATION LOOKUP: the list of Dublin postal districts
# ======================================================================
# This project now focuses on Dublin only, at postal-district level.
# Every Dublin dataset needs the same district names, so this small table
# is made first and used by all the other Dublin notebooks.
#
# Location levels used in the project:
#   nation          -> Ireland
#   province        -> Leinster
#   county          -> Dublin
#   local_authority -> Dublin City / Fingal / South Dublin / Dun Laoghaire-Rathdown
#                      (only the planning data says which council)
#   dublin_district -> Dublin 1 ... Dublin 24, Dublin 6W
#                      or 'Dublin (no district)' for places like Swords or Lucan
#
# Output: just_dublin_dataset\just_dublin_processed_dataset\dublin_location_lookup.csv
# ======================================================================

import pandas as pd
import os

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

national_clean = base + r'\dataSet\clean_dataset'
dublin_processed = base + r'\just_dublin_dataset\just_dublin_processed_dataset'

os.makedirs(dublin_processed, exist_ok=True)

In [2]:
# The official Dublin postal districts.
# Odd numbers are north of the river Liffey, even numbers south.
# There is no Dublin 19, 21 or 23, and Dublin 6W is its own district.

districts = ['Dublin 1', 'Dublin 2', 'Dublin 3', 'Dublin 4', 'Dublin 5', 'Dublin 6',
             'Dublin 6W', 'Dublin 7', 'Dublin 8', 'Dublin 9', 'Dublin 10', 'Dublin 11',
             'Dublin 12', 'Dublin 13', 'Dublin 14', 'Dublin 15', 'Dublin 16', 'Dublin 17',
             'Dublin 18', 'Dublin 20', 'Dublin 22', 'Dublin 24']

print(len(districts), 'postal districts')

22 postal districts


In [3]:
# Each Dublin district has an Eircode area code (the first 3 characters of
# an Eircode): Dublin 1 -> D01, Dublin 12 -> D12, Dublin 6W -> D6W.
# This is how the sales data (PPR) can be matched to a district.

eircode_areas = []
for name in districts:
    number = name.replace('Dublin ', '')      # 'Dublin 12' -> '12'
    if number == '6W':
        eircode_areas.append('D6W')
    elif len(number) == 1:
        eircode_areas.append('D0' + number)   # '1' -> 'D01'
    else:
        eircode_areas.append('D' + number)    # '12' -> 'D12'

for name, code in zip(districts, eircode_areas):
    print(name, '->', code)

Dublin 1 -> D01
Dublin 2 -> D02
Dublin 3 -> D03
Dublin 4 -> D04
Dublin 5 -> D05
Dublin 6 -> D06
Dublin 6W -> D6W
Dublin 7 -> D07
Dublin 8 -> D08
Dublin 9 -> D09
Dublin 10 -> D10
Dublin 11 -> D11
Dublin 12 -> D12
Dublin 13 -> D13
Dublin 14 -> D14
Dublin 15 -> D15
Dublin 16 -> D16
Dublin 17 -> D17
Dublin 18 -> D18
Dublin 20 -> D20
Dublin 22 -> D22
Dublin 24 -> D24


In [4]:
# Building the lookup table. One extra row for the parts of County Dublin
# that have no district number (Swords, Lucan, Malahide, Dun Laoghaire...).

lookup = pd.DataFrame({'dublin_district': districts,
                       'eircode_area': eircode_areas})

no_district = pd.DataFrame({'dublin_district': ['Dublin (no district)'],
                            'eircode_area': [None]})

lookup = pd.concat([lookup, no_district], ignore_index=True)
lookup['district_id'] = range(1, len(lookup) + 1)

lookup

,dublin_district,eircode_area,district_id
0,Dublin 1,D01,1
1,Dublin 2,D02,2
2,Dublin 3,D03,3
3,Dublin 4,D04,4
4,Dublin 5,D05,5
5,Dublin 6,D06,6
6,Dublin 6W,D6W,7
7,Dublin 7,D07,8
8,Dublin 8,D08,9
9,Dublin 9,D09,10


In [5]:
# Adding the higher levels, which are the same for every Dublin row.
# county_id comes from the national counties.csv so it matches the
# national data (Dublin = 6).

counties = pd.read_csv(national_clean + r'\counties.csv')
dublin_id = counties.loc[counties['county'] == 'Dublin', 'county_id'].values[0]

lookup['nation'] = 'Ireland'
lookup['province'] = 'Leinster'
lookup['county'] = 'Dublin'
lookup['county_id'] = dublin_id

lookup = lookup[['district_id', 'dublin_district', 'eircode_area',
                 'county', 'county_id', 'province', 'nation']]
lookup

,district_id,dublin_district,eircode_area,county,county_id,province,nation
0,1,Dublin 1,D01,Dublin,6,Leinster,Ireland
1,2,Dublin 2,D02,Dublin,6,Leinster,Ireland
2,3,Dublin 3,D03,Dublin,6,Leinster,Ireland
3,4,Dublin 4,D04,Dublin,6,Leinster,Ireland
4,5,Dublin 5,D05,Dublin,6,Leinster,Ireland
5,6,Dublin 6,D06,Dublin,6,Leinster,Ireland
6,7,Dublin 6W,D6W,Dublin,6,Leinster,Ireland
7,8,Dublin 7,D07,Dublin,6,Leinster,Ireland
8,9,Dublin 8,D08,Dublin,6,Leinster,Ireland
9,10,Dublin 9,D09,Dublin,6,Leinster,Ireland


In [6]:
lookup.to_csv(dublin_processed + r'\dublin_location_lookup.csv', index=False)
print('Saved', lookup.shape, 'to', dublin_processed + r'\dublin_location_lookup.csv')

Saved (23, 7) to C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\just_dublin_dataset\just_dublin_processed_dataset\dublin_location_lookup.csv
